In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Multiple Input and Multiple Output Channels

Every kernel we have used so far has read one grid of numbers and produced
one. The edge detector of that section looked at a grayscale image
and found one kind of edge. Real images and real networks need more on both
sides. A color photograph stores three numbers per pixel, one *channel* each
for red, green, and blue, so a $224\times 224$ RGB image is a tensor of shape
$3\times 224\times 224$. A boundary between a red and a green region of equal
brightness is visible only when the channels are compared. And a useful layer
detects dozens of patterns at every position (edges at several orientations,
spots, textures), each in a grid of its own. How does one layer read several
input channels and produce several output channels, and what does that cost?

The convolutional layer of that section already allows for this:
each output channel adds up, over all input channels, a separate
$k_\textrm{h}\times k_\textrm{w}$ kernel applied to each. This section
implements that layer on top of the single-channel `corr2d` and counts what
it costs. A layer with $c_\textrm{i}$ input and $c_\textrm{o}$ output channels
has $c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w}$ weights, and its
cost grows with the product $c_\textrm{i} c_\textrm{o}$. The rest of the
section reduces that product: the $1\times 1$ convolution mixes channels
without looking at neighbors, and grouped and depthwise convolutions restrict
which channels are mixed.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Multiple Input Channels

With $c_\textrm{i}$ input channels, the kernel needs one
$k_\textrm{h}\times k_\textrm{w}$ grid of weights for every input channel, so
it becomes a tensor of shape $c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$.
We cross-correlate each input channel with the matching kernel channel and add
the $c_\textrm{i}$ results. Writing $\mathbf{X}_c$ and $\mathbf{K}_c$ for
channel $c$ of the input and of the kernel, and $\mathbf{X}_c \star
\mathbf{K}_c$ for their two-dimensional cross-correlation the equation,
which `corr2d` computes, the output is the single grid

$$\mathbf{Y} = \sum_{c=1}^{c_\textrm{i}} \mathbf{X}_c \star \mathbf{K}_c.$$

Each output value is thus a weighted sum over the same window in every
channel, with $c_\textrm{i} k_\textrm{h} k_\textrm{w}$ products.
the figure works an example with two input channels. The
shaded first output element is
$(1\times1+2\times2+4\times3+5\times4)+(0\times0+1\times1+3\times2+4\times3)=56$.

![Cross-correlation with two input channels: one cross-correlation per channel, then a sum.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-multi-in.svg)

The sum over channels is what lets a kernel compare colors. A $1\times 1$
kernel with weight $+1$ on the red channel and $-1$ on the green channel
outputs red minus green at every pixel: positive in red regions, negative in
green ones, and zero in gray ones, however bright. Where its sign flips
between neighboring pixels, a red region meets a green one. This holds even
when both regions are equally bright, so that a filter applied to the
brightness alone sees no edge there.

The implementation follows the formula: one call of `corr2d` per channel,
then a sum.

In [2]:
def corr2d_multi_in(X, K):
    """Cross-correlate each channel of X with its channel of K and add."""
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

With the input and kernel of the figure, the first output
element should be 56.

In [3]:
X = d2l.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = d2l.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

## Multiple Output Channels

A multi-input kernel produces one output grid: a feature map, which holds one
detector's responses at every position. To detect $c_\textrm{o}$
patterns, the layer uses $c_\textrm{o}$ such kernels, one per output channel,
each of shape $c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$. Stacked,
they form a kernel tensor of shape
$c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$, and
output channel $o$ is

$$\mathbf{Y}_o = \sum_{c=1}^{c_\textrm{i}} \mathbf{X}_c \star \mathbf{K}_{o,c}, \quad o = 1, \ldots, c_\textrm{o}.$$

Every output channel reads every input channel, each through a kernel of its
own. This is the layer the equation of
that section, written channel by channel: $\mathbf{K}_{o,c}$ is its
$k_\textrm{h}\times k_\textrm{w}$ slice for input channel $c$ and output
channel $o$ (there called $d$). Networks usually increase the number of
channels as strides or pooling reduce the resolution, trading spatial detail
for more kinds of features at each position. The channels are trained
jointly, so one feature, such as an edge at some orientation, may be spread
over several channels instead of occupying one.

PyTorch stores this kernel as
$c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$ and
its layers take channels-first inputs, (example, channel, height, width).

The implementation applies `corr2d_multi_in` once per output channel and
stacks the results.

In [4]:
def corr2d_multi_in_out(X, K):
    """Apply corr2d_multi_in once per output channel of K and stack."""
    return d2l.stack([corr2d_multi_in(X, k) for k in K], 0)

To test it, we build a kernel with three output channels from `K`, `K + 1`,
and `K + 2`.

In [5]:
K = d2l.stack((K, K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

The output has three channels.

In [6]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

The first channel reproduces the output above, because its kernel is the
original `K`. Adding 1 to every weight adds to each output the sum of its
window over both input channels, which is 20 for the first element, so the
second channel starts at $56 + 20 = 76$.

### Parameters and Cost

Channels multiply the size of a layer, and before we design networks with
many of them we need to know by how much. With one bias per output channel, a
convolutional layer has

$$c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w} + c_\textrm{o}$$

parameters. Each output value takes one multiply-add per weight of its output
channel, $c_\textrm{i} k_\textrm{h} k_\textrm{w}$ in total, so a layer with an
$h_\textrm{o}\times w_\textrm{o}$ output performs

$$h_\textrm{o} w_\textrm{o} \cdot c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w}$$

multiply-adds. Both counts grow with the product $c_\textrm{i} c_\textrm{o}$:
doubling the number of channels in every layer quadruples them. The cost, in
addition, grows with the number of output positions, since the same weights
are applied at each. The first layer from 3 to 64 channels with $3\times 3$
kernels in that section has 1,792 parameters. A layer deeper in,
from 256 to 256 channels with $3\times 3$ kernels, has 590,080 parameters,
and on a $56\times 56$ output it performs
$56^2\cdot 589{,}824 \approx 1.8\times 10^9$ multiply-adds for a single
image. Training also keeps each layer's output,
$h_\textrm{o} w_\textrm{o} c_\textrm{o}$ values per example, for the backward
pass (that section). This activation memory is
largest in the early layers, where the resolution is high and the weights are few:
the 3-to-64-channel layer with a $224\times 224$ output keeps
$224^2\cdot 64 \approx 3.2\times 10^6$ values per example, against its 1,792
parameters. Reducing the factor $c_\textrm{i} c_\textrm{o}$ is the subject of
the rest of the section.

## The $1\times 1$ Convolution

A $1\times 1$ kernel, $k_\textrm{h} = k_\textrm{w} = 1$, looks at a single
pixel and cannot detect spatial patterns. With several channels it still
computes something useful. At every position it maps the $c_\textrm{i}$ input
values to $c_\textrm{o}$ output values with the same
$c_\textrm{o}\times c_\textrm{i}$ matrix of weights: it is a fully connected
layer applied to each pixel separately, with weights shared across pixels.
It has $c_\textrm{o} c_\textrm{i}$ weights plus $c_\textrm{o}$ biases.
the figure shows a $1\times 1$ convolution from 3 input channels
to 2 output channels; input and output have the same height and width.

![A $1\times 1$ convolution with three input channels and two output channels. Each output element is a linear combination of the input elements at the same position.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-1x1.svg)

Because the same matrix acts at every pixel, the whole layer is one matrix
product. We flatten the spatial axes of the input into a
$c_\textrm{i}\times hw$ matrix, multiply it by the kernel reshaped to
$c_\textrm{o}\times c_\textrm{i}$, and restore the spatial axes.

In [7]:
def corr2d_multi_in_out_1x1(X, K):
    """A 1x1 convolution as one matrix product."""
    c_i, h, w = X.shape
    c_o = K.shape[0]
    Y = d2l.matmul(d2l.reshape(K, (c_o, c_i)), d2l.reshape(X, (c_i, h * w)))
    return d2l.reshape(Y, (c_o, h, w))

On random data the matrix product agrees with `corr2d_multi_in_out`.

In [8]:
X = d2l.normal(0, 1, (3, 3, 3))
K = d2l.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
float(d2l.reduce_sum(d2l.abs(Y1 - Y2)))

0.0

The total absolute difference over the 18 outputs is zero or at most about
$10^{-6}$, the size of rounding errors in 32-bit floating-point arithmetic.

A $1\times 1$ convolution changes the number of channels cheaply, and deep
networks use it to reduce the cost of the spatial convolutions around it.
Take the $3\times 3$ layer on 256 channels from above. A $1\times 1$
convolution first reduces 256 channels to 64, the $3\times 3$ convolution
then works on 64 channels, and a second $1\times 1$ convolution restores 256.
The three layers have $16{,}384 + 36{,}864 + 16{,}384 = 69{,}632$ weights
against $589{,}824$, about 8.5 times fewer, and the multiply-adds fall by the
same factor. Such a *bottleneck* can only represent maps that pass through 64
channels, a restriction that networks such as ResNet accept for the savings.
A nonlinearity after a $1\times 1$ convolution makes it a small nonlinear
network applied at every pixel [@Lin.Chen.Yan.2013]. Without one, it can
be folded into the next convolution, since two linear maps compose to one.

## Grouped, Depthwise, and Depthwise-Separable Convolutions

### Grouped Convolution

The factor $c_\textrm{i} c_\textrm{o}$ arises because every output channel
reads every input channel. A *grouped convolution* splits the input channels
into $g$ groups of $c_\textrm{i}/g$ channels and the output channels into $g$
groups of $c_\textrm{o}/g$, and lets each output group read only its own
input group. the figure shows which input channels each output
channel reads. The dense pattern becomes block diagonal, with $g$ blocks of
size $(c_\textrm{o}/g)\times(c_\textrm{i}/g)$, and parameters and
multiply-adds fall from $c_\textrm{i} c_\textrm{o} k_\textrm{h} k_\textrm{w}$
to

$$g\cdot\frac{c_\textrm{i}}{g}\cdot\frac{c_\textrm{o}}{g}\, k_\textrm{h} k_\textrm{w} = \frac{c_\textrm{i} c_\textrm{o} k_\textrm{h} k_\textrm{w}}{g},$$

a factor of $g$. In exchange, no information flows between groups within the
layer. Networks built from grouped convolutions, such as ResNeXt
[@Xie.Girshick.Dollar.ea.2017], restore the exchange with
$1\times 1$ convolutions between grouped layers.

![Input channels read by each output channel, for 8 input and 8 output channels. Each shaded cell is one $k_\textrm{h}\times k_\textrm{w}$ kernel; $g$ groups keep $1/g$ of them.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-groups.svg)

### Depthwise-Separable Convolution

With $g = c_\textrm{i} = c_\textrm{o}$, every group holds one channel. This
*depthwise convolution* filters each channel with its own $k\times k$ kernel
and mixes no channels at all, the opposite of the $1\times 1$ convolution,
which mixes channels and ignores neighbors. A depthwise $k\times k$
convolution followed by a *pointwise* $1\times 1$ convolution is a
*depthwise-separable convolution*
[@chollet2017xception; @howard2017mobilenet].
the figure contrasts it with the dense layer it replaces:
the depthwise stage looks at neighborhoods within each channel, and the
pointwise stage recombines the channels at each position.

![Top: a dense convolution, in which every output channel reads every input channel. Bottom: a depthwise convolution filters each channel separately, and a pointwise $1\times 1$ convolution mixes the channels.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-conv-depthwise.svg)

For a $k\times k$ kernel and an $h\times w$ output, the dense convolution
performs $hw\cdot c_\textrm{i} c_\textrm{o} k^2$ multiply-adds. The separable
pair performs $hw\cdot c_\textrm{i} k^2$ in the depthwise stage and
$hw\cdot c_\textrm{i} c_\textrm{o}$ in the pointwise stage, so the cost
relative to the dense layer is

$$
\frac{h w \cdot c_\textrm{i} k^2 + h w \cdot c_\textrm{i} c_\textrm{o}}{h w \cdot c_\textrm{i} c_\textrm{o} k^2}
= \frac{1}{c_\textrm{o}} + \frac{1}{k^2}.
$$

The same ratio holds for the weights. For $k = 3$ and many output channels it
is close to $1/9$: the separable layer is nearly nine times cheaper.

The savings come from a restriction. The separable pair applies the
depthwise filter $\mathbf{D}_c$ to input channel $c$ and scales the result by
the pointwise weight $P_{o,c}$ before adding it to output channel $o$. It is
therefore the dense convolution with kernel
$\mathbf{K}_{o,c} = P_{o,c}\,\mathbf{D}_c$: all output channels see the same
spatial filter of each input channel, only scaled, while a dense layer may
use a different filter for every pair $(o, c)$. Efficient architectures accept
this restriction and spend the savings on more channels or more layers, as
that section shows.

We check the arithmetic with a dense $3\times 3$ convolution from 128 to 128
channels and its depthwise-separable factorization, without biases. Both map
the input to an output of the same shape, and
the equation predicts a ratio of parameter counts of
$(1/128 + 1/9)^{-1} \approx 8.4$.

In [9]:
c_i, c_o, k = 128, 128, 3
X = d2l.normal(0, 1, (1, c_i, 32, 32))
dense = nn.LazyConv2d(c_o, kernel_size=k, padding=1, bias=False)
depthwise = nn.LazyConv2d(c_i, kernel_size=k, padding=1, groups=c_i,
                          bias=False)
pointwise = nn.LazyConv2d(c_o, kernel_size=1, bias=False)
assert dense(X).shape == pointwise(depthwise(X)).shape
p_dense = dense.weight.numel()
p_sep = depthwise.weight.numel() + pointwise.weight.numel()
p_dense, p_sep, p_dense / p_sep

(147456, 17536, 8.408759124087592)

The dense layer has 147,456 weights and the separable pair 17,536, a ratio of
8.4 as predicted.

In PyTorch, `groups=c_i` turns `nn.Conv2d` into a depthwise convolution;
any `groups` that divides both channel counts gives a grouped convolution.

We can now build convolutional layers with any number of input and output
channels, count their parameters and multiply-adds, and reduce the cost of
mixing channels with $1\times 1$, grouped, and depthwise-separable layers.
The output of every such layer still shifts with its input pixel by pixel,
and that section summarizes neighborhoods so that small shifts of
the input change the output little.

## Summary

With several input channels, a convolution cross-correlates each input
channel with its own kernel and adds the results, which lets one kernel
combine evidence across channels such as colors. With several output
channels, the layer holds one such kernel per output channel, a tensor of
shape $c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$
with $c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w} + c_\textrm{o}$
parameters and $h_\textrm{o} w_\textrm{o} c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w}$
multiply-adds, so the cost grows with the product of the channel counts. A
$1\times 1$ convolution is a fully connected layer applied at every pixel and
changes the number of channels cheaply, as in bottlenecks. Grouped
convolutions divide the cost by the number of groups, and a depthwise
convolution followed by a $1\times 1$ convolution reduces it to a fraction
$1/c_\textrm{o} + 1/k^2$, at the price of a restricted, factored kernel.

## Exercises

1. **Composition of convolutions.** Two convolutional layers with stride
   1, no padding, and no nonlinearity in between have kernels of shape
   $c_\textrm{m} \times c_\textrm{i} \times k_1 \times k_1$ and
   $c_\textrm{o} \times c_\textrm{m} \times k_2 \times k_2$.
    1. Prove that their composition is a single convolution, and give the
       shape of its kernel.
    1. Compare the number of weights of the pair with that of the single
       convolution. For which intermediate width $c_\textrm{m}$ is the
       pair cheaper?
    1. As that section showed, three stacked single-channel
       $3 \times 3$ kernels have a $7 \times 7$ receptive field. Decide
       whether every single-channel $7 \times 7$ kernel is the composition
       of three $3 \times 3$ kernels, and justify the answer.

1. **Cost and memory accounting.** A convolution with kernel shape
   $c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$,
   total padding $(p_\textrm{h}, p_\textrm{w})$, and stride
   $(s_\textrm{h}, s_\textrm{w})$ is applied to an input of shape
   $c_\textrm{i}\times h\times w$. Let $h_\textrm{o}\times w_\textrm{o}$
   be the output shape given by the equation with
   $n_\textrm{h} = h$, $n_\textrm{w} = w$, and dilation 1.
    1. Starting from the multiply-add count of this section, count the
       multiplications and the additions of the forward propagation
       separately, bias included, in terms of $h_\textrm{o}$ and
       $w_\textrm{o}$. How does the count change when the padding is
       doubled?
    1. How many values do the input, the kernel, and the output occupy?
       Which of these tensors must the forward propagation keep for the
       backward propagation?
    1. Count the multiplications needed for the gradients with respect
       to the kernel and with respect to the input, and give the number
       of values these gradients occupy.

1. [code] **Multi-channel im2col.** Extend the patch matrix of
   that section to several input and output channels.
   Rebuild the two-channel `X` of the figure and the
   three-output-channel `K` of that section,
   since later cells of this section overwrite both. Build the patch
   matrix, whose rows are input patches of shape
   $c_\textrm{i} \times k_\textrm{h} \times k_\textrm{w}$ flattened in
   that order, and arrange `K` into a
   $c_\textrm{i} k_\textrm{h} k_\textrm{w} \times c_\textrm{o}$ kernel
   matrix whose rows follow the same order. Confirm that their product,
   rearranged into the shape of the output, reproduces
   `corr2d_multi_in_out(X, K)`.

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problems 10.6 and 10.7.*

1. **Strip-buffered convolution.** A fast implementation of a stride-1
   $k \times k$ convolution copies a strip of input columns, spanning all
   $c_\textrm{i}$ channels and the full height, into fast memory. It
   computes output columns from the strip and then moves the strip right
   by as many columns as it computed. One scheme loads $k$ columns and
   computes one output column. The alternative loads $k + m - 1$ columns
   and computes $m$ output columns. Both schemes load their whole strip
   anew at every step.
    1. On average, how many times is each input column loaded under each
       scheme?
    1. What limits how large $m$ can be, and how does the load count of
       the previous sub-item change as $m$ grows?

1. **Depthwise-separable block.** Consider a block of two dense
   $3 \times 3$ convolutions, each with $c$ input and $c$ output channels,
   the building block of VGG (that section). Now replace each of the
   two convolutions by its depthwise-separable counterpart. Use no biases
   and padding that preserves the spatial size.
    1. Compute the number of parameters and the number of multiplications
       on an $h \times w$ input for both variants. Which stage of the
       separable block, depthwise or pointwise, accounts for most of these
       counts, and from which $c$ on?
    1. Give the separable block the same parameter budget as the dense
       block and let it use $c'$ channels instead of $c$. Solve for $c'$
       and state its limit relative to $c$ as $c$ grows. What does this
       suggest about how to spend the savings of the factorization?

1. **Channel permutation between groups.** ● Grouped layers can also be
   mixed without dense weights. Follow a grouped $k \times k$ convolution
   from $c_\textrm{i}$ to $c_\textrm{o}$ channels with $g$ groups by a
   second grouped convolution from $c_\textrm{o}$ to $c_\textrm{o}$
   channels with $g$ groups, and insert a fixed permutation of the
   $c_\textrm{o}$ intermediate channels between the two layers. Find the
   condition on $g$ and $c_\textrm{o}$ under which some permutation makes
   each output of the second layer depend on all $c_\textrm{i}$ input
   channels, and construct such a permutation when the condition holds.

[Discussions](https://d2l.discourse.group/t/70)